# 🛠️ Aula 06 — Primeira Ferramenta
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula06_primeira_ferramenta_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Ferramentas com `@tool` do LangChain: calculadora e horário. O LLM decide quando usar.

---


## 🏗️ Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

Execute a célula abaixo e avance — é boilerplate reutilizável.

In [ ]:
# ── Setup completo: Ollama + gemma4:e2b + warm up ────────────────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain-openai langchain-core requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelo
!ollama pull gemma4:e2b

# 6. Warm up (primeira inferência demora ~2-3 min)
print("🔥 Warm up...")
start = time.time()
!curl -s http://localhost:11434/api/chat \\
    -d '{"model":"gemma4:e2b","messages":[{"role":"user","content":"Oi"}],"stream":false,"keep_alive":-1}' \\
    > /dev/null
print(f"✅ Pronto em {time.time()-start:.1f}s")


## 1. Ferramentas com `@tool`

`@tool` transforma uma função Python em ferramenta que o LLM pode invocar.
O decorador extrai nome, descrição e tipos dos parâmetros automaticamente.

In [ ]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def calcular(a: float, b: float, operacao: str) -> float:
    """Realiza uma operação matemática entre dois números.
    operacao pode ser: soma, subtracao, multiplicacao, divisao."""
    ops = {
        "soma": a + b,
        "subtracao": a - b,
        "multiplicacao": a * b,
        "divisao": a / b if b != 0 else float("inf"),
    }
    return ops.get(operacao.lower(), float("nan"))

@tool
def obter_horario() -> str:
    """Retorna a data e hora atual."""
    return datetime.now().strftime("%d/%m/%Y %H:%M:%S")

print("✅ Ferramentas registradas:", [t.name for t in [calcular, obter_horario]])

## 2. Bind Tools no LLM

`llm.bind_tools()` registra as ferramentas no modelo. O LLM decide quando chamá-las.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gemma4:e2b",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",
    temperature=0.3
)

llm_com_ferramentas = llm.bind_tools([calcular, obter_horario])
print("✅ Tools bindadas:", [t["function"]["name"] for t in llm_com_ferramentas.kwargs["tools"])

## 3. Demo — O LLM decide quando usar ferramentas

In [ ]:
# Pergunta que precisa de calculadora
resposta1 = llm_com_ferramentas.invoke("Quanto é 234 vezes 987?")
print("📝 Resposta do LLM:")
print(f"  content: {resposta1.content}")
print(f"  tool_calls: {resposta1.tool_calls}")

In [ ]:
# Executar a tool_call manualmente
tc = resposta1.tool_calls[0]
resultado = calcular.invoke({"a": tc["args"]["a"], "b": tc["args"]["b"], "operacao": tc["args"]["operacao"]})
print(f"🔧 Ferramenta executada: {tc['name']}({tc['args']}) = {resultado}")

In [ ]:
# Pergunta sobre horário
resposta2 = llm_com_ferramentas.invoke("Que horas são?")
print("📝 Resposta do LLM:")
print(f"  content: {resposta2.content}")
print(f"  tool_calls: {resposta2.tool_calls}")

if resposta2.tool_calls:
    tc2 = resposta2.tool_calls[0]
    resultado2 = obter_horario.invoke({})
    print(f"🔧 Horário atual: {resultado2}")

In [ ]:
# Pergunta que NÃO precisa de ferramenta
resposta3 = llm_com_ferramentas.invoke("Qual a capital da França?")
print("📝 Resposta do LLM:")
print(f"  content: {resposta3.content}")
print(f"  tool_calls: {resposta3.tool_calls}")
print("👉 Sem tool_calls — o LLM respondeu sozinho!")

---
✅ Resumo:
- **`@tool`** transforma funções Python em ferramentas para o LLM
- **`bind_tools()`** registra as ferramentas no modelo
- O LLM decide **quando** chamar uma ferramenta via `tool_calls`
- Executamos a ferramenta manualmente a partir de `tool_calls[0]`
- Perguntas comuns → o LLM responde sem ferramenta

Na S07 vamos automatizar o loop de execução com **ToolNode** e **ReAct**!

*Material da Guilda de IA — UFVJM 2026.1*